# MoP + DivPO Co-Author SFT Training

Trains one LoRA adapter per writing-partner persona using the co-author SFT format.
Target behavior: structured pre-writing help — diagnosis, idea, rationale, next step.

Before running, push local data to HuggingFace:
```bash
PYTHONPATH=src python3 -m mop_divpo.cli build-coauthor-data --output-dir data/processed/coauthor_sft
```
Then upload to `DasonTio/mop-divpo-coauthor-sft-data` (already done if using this notebook).

In [ ]:
# Cell 1 - Install dependencies
!pip install -q trl peft transformers datasets accelerate huggingface_hub pydantic

In [ ]:
# Cell 2 - Check GPU
import torch

assert torch.cuda.is_available(), 'No GPU detected. Set Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 3 - HuggingFace login (needed to push adapters)
from huggingface_hub import login

login()

In [ ]:
# Cell 4 - Load co-author SFT data from HuggingFace
# Option A (default): load from HF dataset repo
# Option B: upload JSONL files manually into /content/coauthor_sft and skip this cell
import json
from pathlib import Path
from datasets import load_dataset

PERSONA_IDS = ('contrarian', 'cross_domain_analogist', 'systems_thinker', 'minimalist')
HF_DATASET = 'DasonTio/mop-divpo-coauthor-sft-data'
DATA_DIR = Path('/content/coauthor_sft')
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    ds = load_dataset(HF_DATASET)
    for persona, split in ds.items():
        out = DATA_DIR / f'{persona}.jsonl'
        with out.open('w') as f:
            for row in split:
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
        print(f'{persona}: {len(split)} rows -> {out}')
except Exception as exc:
    print('Could not load HF dataset. Upload JSONL files manually to', DATA_DIR)
    print(type(exc).__name__, exc)

In [ ]:
# Cell 5 - Training utilities
import json
from pathlib import Path
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
HF_ADAPTER_PREFIX = 'DasonTio/mop-divpo-coauthor'


def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def format_records(records, tokenizer):
    formatted = []
    for row in records:
        messages = row.get('messages')
        if not isinstance(messages, list) or len(messages) < 3:
            continue
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted.append({'text': text})
    return formatted


def train_persona(persona, data_path, output_dir, max_steps=200, push_to_hub=True):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    records = read_jsonl(data_path)
    formatted = format_records(records, tokenizer)
    assert formatted, f'No usable co-author message rows in {data_path}'
    print(persona, 'training rows:', len(formatted))

    dataset = Dataset.from_list(formatted)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map='auto')

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=['q_proj', 'v_proj'],
        bias='none',
    )
    args = SFTConfig(
        output_dir=str(output_dir),
        max_steps=max_steps,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=max_steps,
        save_total_limit=1,
        report_to='none',
        dataset_text_field='text',
    )
    trainer = SFTTrainer(model=model, args=args, train_dataset=dataset, peft_config=lora_config)
    trainer.train()
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    if push_to_hub:
        repo_id = f'{HF_ADAPTER_PREFIX}-{persona}-sft'
        trainer.model.push_to_hub(repo_id)
        tokenizer.push_to_hub(repo_id)
        print('Pushed:', repo_id)

    del model, trainer
    torch.cuda.empty_cache()

In [ ]:
# Cell 6 - Train all personas
OUTPUT_DIR = Path('/content/coauthor_adapters')
MAX_STEPS = 200

for persona in PERSONA_IDS:
    data_path = DATA_DIR / f'{persona}.jsonl'
    if not data_path.exists():
        print('Skipping missing data:', data_path)
        continue
    print(f'\n=== Training {persona} ===')
    train_persona(persona, data_path, OUTPUT_DIR / persona, max_steps=MAX_STEPS, push_to_hub=True)

print('Co-author adapter training complete.')

In [ ]:
# Cell 7 - Smoke test one adapter as a writing partner
from peft import PeftModel

PERSONA = 'contrarian'
ADAPTER_REPO = f'{HF_ADAPTER_PREFIX}-{PERSONA}-sft'
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map='auto')
model = PeftModel.from_pretrained(base, ADAPTER_REPO)
model.eval()

messages = [
    {'role': 'system', 'content': 'You are a Contrarian Analyst and writing partner. Help the writer challenge hidden assumptions and find sharper thesis angles.'},
    {'role': 'user', 'content': 'Topic: AI in education\nWriter goal: Find a non-obvious thesis angle\nAudience: university instructors'},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.8, top_p=0.9, pad_token_id=tokenizer.eos_token_id)

print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))